In [6]:
import sqlite3
print(sqlite3.sqlite_version)
import pandas as pd


3.45.3


In [15]:
import sys
print(sys.version)

3.9.19 (main, May  6 2024, 20:12:36) [MSC v.1916 64 bit (AMD64)]


In [30]:
# Database Setup
def initialize_db():
    conn = sqlite3.connect("inventory.db")
    cursor = conn.cursor()
    
    # User roles
    cursor.execute("DROP TABLE IF EXISTS users")
    cursor.execute("DROP TABLE IF EXISTS inventory")
    cursor.execute("DROP TABLE IF EXISTS orders")
    cursor.execute("DROP TABLE IF EXISTS waste_log")
    cursor.execute(
        """CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            password TEXT NOT NULL,
            role TEXT CHECK(role IN ('Admin', 'Manager', 'Staff')) NOT NULL
        );""")
    
    # Inventory Table
    cursor.execute(
        """CREATE TABLE IF NOT EXISTS inventory (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            quantity INTEGER NOT NULL,
            threshold INTEGER NOT NULL
        );""")
    
    # Orders Table
    cursor.execute(
        """CREATE TABLE IF NOT EXISTS orders (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            item_name TEXT NOT NULL,
            quantity INTEGER NOT NULL,
            status TEXT DEFAULT 'Pending'
        );""")
    
    # Waste Log Table
    cursor.execute(
        """CREATE TABLE IF NOT EXISTS waste_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            item_name TEXT NOT NULL,
            quantity INTEGER NOT NULL,
            reason TEXT NOT NULL
        );"""
    )
    add_user(test_username, test_password, test_role)
    
    conn.commit()
    conn.close()

# Add User
def add_user(username, password, role):
    conn = sqlite3.connect("inventory.db")
    cursor = conn.cursor()
    
    try:
        cursor.execute("""
            INSERT INTO users (username, password, role)
            VALUES (?, ?, ?)
        """, (username, password, role))

        conn.commit()
        print("User added successfully!")

    except sqlite3.IntegrityError:
        print("Error: Username already exists.")

    finally:
        conn.close()

# Example usage
test_username = "admin"
test_password = "password123"
test_role = "Admin"

# User Authentication
def authenticate_user(username, password):
    conn = sqlite3.connect("inventory.db")
    cursor = conn.cursor()
    cursor.execute("SELECT role FROM users WHERE username = ? AND password = ?", (username, password))
    user = cursor.fetchone()
    conn.close()
    return user[0] if user else None

# Add Item to Inventory
def add_inventory_item(name, quantity, threshold):
    conn = sqlite3.connect("inventory.db")
    cursor = conn.cursor()
    cursor.execute("INSERT INTO inventory (name, quantity, threshold) VALUES (?, ?, ?)", (name, quantity, threshold))
    conn.commit()
    conn.close()
    print(f"Added '{name}' with quantity {quantity} and threshold {threshold}.")

# View Inventory
def view_inventory():
    conn = sqlite3.connect("inventory.db")
    df = pd.read_sql_query("SELECT * FROM inventory", conn)
    conn.close()
    print(df)

# Check Low Stock & Place Order
def check_low_stock():
    conn = sqlite3.connect("inventory.db")
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM inventory WHERE quantity < threshold")
    low_stock_items = cursor.fetchall()
    conn.close()
    
    if low_stock_items:
        print("Low stock alert!")
        for item in low_stock_items:
            print(f"{item[1]} - {item[2]} units left (Threshold: {item[3]})")
            place_order(item[1], item[3] - item[2])
    else:
        print("All stock levels are sufficient.")

# Place Order
def place_order(item_name, quantity):
    conn = sqlite3.connect("inventory.db")
    cursor = conn.cursor()
    cursor.execute("INSERT INTO orders (item_name, quantity) VALUES (?, ?)", (item_name, quantity))
    conn.commit()
    conn.close()
    print(f"Order placed for {quantity} units of {item_name}.")

# View Orders
def view_orders():
    conn = sqlite3.connect("inventory.db")
    df = pd.read_sql_query("SELECT * FROM orders", conn)
    conn.close()
    print(df)

# Log Waste
def log_waste(item_name, quantity, reason):
    conn = sqlite3.connect("inventory.db")
    cursor = conn.cursor()
    cursor.execute("INSERT INTO waste_log (item_name, quantity, reason) VALUES (?, ?, ?)", (item_name, quantity, reason))
    conn.commit()
    conn.close()
    print(f"Logged waste: {item_name}, {quantity} units ({reason}).")

# Main Menu
def main(username, password):
    initialize_db()
    role = authenticate_user(username, password)
    
    if role:
        print(f"Welcome, {username} ({role})!")
    else:
        print("Authentication failed!")
        return
    
    while True:
        print("\n--- Inventory Management System ---")
        print("1. Add Inventory Item")
        print("2. View Inventory")
        print("3. Check Low Stock")
        print("4. View Orders")
        print("5. Log Waste")
        print("6. Exit")
        choice = input("Select an option (1-6): ")
        
        if choice == "1" and role in ['Admin', 'Manager']:
            name = input("Item Name: ")
            quantity = int(input("Initial Quantity: "))
            threshold = int(input("Low Stock Threshold: "))
            add_inventory_item(name, quantity, threshold)
        elif choice == "2":
            view_inventory()
        elif choice == "3" and role in ['Admin', 'Manager']:
            check_low_stock()
        elif choice == "4" and role in ['Admin', 'Manager']:
            view_orders()
        elif choice == "5" and role in ['Staff', 'Manager']:
            name = input("Item Name: ")
            quantity = int(input("Quantity Wasted: "))
            reason = input("Reason for Waste: ")
            log_waste(name, quantity, reason)
        elif choice == "6":
            print("Exiting...")
            break
        else:
            print("Invalid choice or insufficient permissions. Try again.")

test_username = "admin"
test_password = "password123"
main(test_username, test_password)

User added successfully!
Welcome, admin (Admin)!

--- Inventory Management System ---
1. Add Inventory Item
2. View Inventory
3. Check Low Stock
4. View Orders
5. Log Waste
6. Exit


Select an option (1-6):  6


Exiting...


In [28]:
# User Authentication
def authenticate_user(username, password):
    conn = sqlite3.connect("inventory.db")
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM users WHERE username = 'admin';")
    cursor.execute("SELECT role FROM users WHERE username = ? AND password = ?", (username, password))
    user = cursor.fetchone()
    conn.close()
    return user[0] if user else None
print(authenticate_user("adimin","password123"))

None


In [29]:
def get_user(username):
    """Fetch user details from the database."""
    conn = sqlite3.connect("inventory.db")  # Connect to the database
    cursor = conn.cursor()

    # Execute SQL query to fetch user details
    cursor.execute("SELECT * FROM users WHERE username = ?", (username,))
    user = cursor.fetchone()  # Fetch the first matching row

    conn.close()  # Close the connection
    return user

# Run query for username 'admin'
user_data = get_user("admin")

# Print results
if user_data:
    print("User found:", user_data)
else:
    print("User not found!")


User not found!


In [ ]:
# Add Item to Inventory
def add_inventory_item(name, quantity, threshold):
    conn = sqlite3.connect("inventory.db")
    cursor = conn.cursor()
    cursor.execute("INSERT INTO inventory (name, quantity, threshold) VALUES (?, ?, ?)", (name, quantity, threshold))
    conn.commit()
    conn.close()
    print(f"Added '{name}' with quantity {quantity} and threshold {threshold}.")

# View Inventory
def view_inventory():
    conn = sqlite3.connect("inventory.db")
    df = pd.read_sql_query("SELECT * FROM inventory", conn)
    conn.close()
    print(df)

In [ ]:
# Check Low Stock & Place Order
def check_low_stock():
    conn = sqlite3.connect("inventory.db")
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM inventory WHERE quantity < threshold")
    low_stock_items = cursor.fetchall()
    conn.close()
    
    if low_stock_items:
        print("Low stock alert!")
        for item in low_stock_items:
            print(f"{item[1]} - {item[2]} units left (Threshold: {item[3]})")
            place_order(item[1], item[3] - item[2])
    else:
        print("All stock levels are sufficient.")

In [ ]:
# Place Order
def place_order(item_name, quantity):
    conn = sqlite3.connect("inventory.db")
    cursor = conn.cursor()
    cursor.execute("INSERT INTO orders (item_name, quantity) VALUES (?, ?)", (item_name, quantity))
    conn.commit()
    conn.close()
    print(f"Order placed for {quantity} units of {item_name}.")

In [ ]:
# View Orders
def view_orders():
    conn = sqlite3.connect("inventory.db")
    df = pd.read_sql_query("SELECT * FROM orders", conn)
    conn.close()
    print(df)